# Étude — Pourquoi la VaR inconditionnelle a échoué en mars 2020 (et comment la volatilité conditionnelle corrige)

**Question.** Le backtest de la brique 0 rejette le test de Christoffersen partout : les exceptions
de VaR arrivent en grappes. Une VaR estimée sur une fenêtre glissante de 250 jours suppose la
volatilité constante sur la fenêtre — que se passe-t-il quand le régime change brutalement (COVID,
mars 2020) et est-ce que remplacer σ par σ_t (EWMA λ=0.94, GARCH(1,1) MLE maison) corrige le défaut ?

**Protocole.** VaR 99 % 1 jour, out-of-sample, sur les dates 2019-01 → 2021-12 (751 points),
portefeuille de référence 8 titres (données : snapshot figé `data/cache/`, 2026-07-06).
Quatre modèles : historique 250 j, paramétrique gaussienne 250 j (inconditionnels),
EWMA RiskMetrics, GARCH(1,1) réestimé tous les 20 j sur fenêtre 1000 j (conditionnels).

**Conclusion (résumé).** La volatilité conditionnelle **répare la dynamique** : le test
d'indépendance de Christoffersen passe (p ≈ 0.61 contre 0.005 pour la paramétrique — plus de
grappes). Mais elle **ne répare pas le niveau** : Kupiec rejette toujours (21 exceptions pour
7.5 attendues), car les innovations restent gaussiennes alors que les résidus standardisés sont
leptokurtiques. → Brique 2 : innovations Student-t. Les verdicts de cette étude sont verrouillés
par `tests/test_etude_2020.py`.


In [1]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from riskplatform.backtest import (
    christoffersen_cc, christoffersen_independence, count_exceptions, kupiec_pof,
)
from riskplatform.config import load_config
from riskplatform.data import load_returns
from riskplatform.portfolio import portfolio_returns
from riskplatform.var import rolling_var, rolling_var_conditional
from riskplatform.volatility import ewma_volatility, fit_garch, garch_variance

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ASSETS = REPO / "assets"
ALPHA = 0.99
STUDY = slice("2019-01-01", "2021-12-31")

config = load_config(REPO / "config" / "portfolio.yaml")
tickers = list(config.portfolio.weights.index)
_, returns = load_returns(
    tickers, config.portfolio.currencies, config.start, config.end,
    cache_dir=REPO / "data" / "cache",
)
pnl = portfolio_returns(returns, config.portfolio.weights)
print(f"{len(pnl)} rendements quotidiens, {pnl.index[0].date()} -> {pnl.index[-1].date()}")


3112 rendements quotidiens, 2014-01-03 -> 2026-07-02


## 1. Les quatre modèles de VaR, backtestés sur 2019–2021

Convention commune (SPEC.md B1.0) : σ²_t est la prévision pour le jour t construite avec les
rendements jusqu'à t-1 — aucun modèle ne voit r_t dans sa VaR du jour t.


In [2]:
var_models = {
    "historique 250j": rolling_var(pnl, "historical", alpha=ALPHA, window=250),
    "paramétrique 250j": rolling_var(pnl, "parametric", alpha=ALPHA, window=250),
    "EWMA (lambda=0.94)": rolling_var_conditional(pnl, "ewma", alpha=ALPHA),
    "GARCH(1,1) refit 20j": rolling_var_conditional(
        pnl, "garch", alpha=ALPHA, window=1000, refit_every=20
    ),
}

rows = []
exceptions_by_model = {}
for name, var_series in var_models.items():
    var_study = var_series.loc[STUDY]
    exc = count_exceptions(pnl.loc[var_study.index], var_study)
    exceptions_by_model[name] = exc
    kupiec = kupiec_pof(exc, alpha=ALPHA)
    ind = christoffersen_independence(exc)
    cc = christoffersen_cc(exc, alpha=ALPHA)
    rows.append({
        "modèle": name,
        "exceptions": kupiec["n_exceptions"],
        "attendues": round(kupiec["expected"], 1),
        "Kupiec p": round(kupiec["p_value"], 4),
        "Kupiec": "REJET" if kupiec["reject"] else "ok",
        "Indép. p": round(ind["p_value"], 4),
        "Indép.": "REJET" if ind["reject"] else "ok",
        "CC p": round(cc["p_value"], 5),
        "CC": "REJET" if cc["reject"] else "ok",
    })

verdicts = pd.DataFrame(rows).set_index("modèle")
verdicts


,exceptions,attendues,Kupiec p,Kupiec,Indép. p,Indép.,CC p,CC
modèle,,,,,,,,
historique 250j,10,7.5,0.3848,ok,0.1172,ok,0.20091,ok
paramétrique 250j,17,7.5,0.0028,REJET,0.0048,REJET,0.00022,REJET
EWMA (lambda=0.94),21,7.5,0.0000,REJET,0.6135,ok,0.00024,REJET
"GARCH(1,1) refit 20j",21,7.5,0.0000,REJET,0.6135,ok,0.00024,REJET


**Lecture.**
- **Paramétrique 250 j** : rejetée partout. 17 exceptions dont 9 concentrées entre le 24 février
  et le 1er avril 2020 — la fenêtre de 250 jours, calibrée sur le calme de 2019, ne « voit » le
  choc qu'avec des mois de retard.
- **EWMA et GARCH** : l'indépendance passe (p ≈ 0.61). Dans la crise, seuls les 4 premiers jours
  de choc (24/02, 27/02, 09/03, 12/03) font exception ; ensuite la VaR a été multipliée par ~3
  et suit le régime. Mais Kupiec rejette : 21 exceptions au lieu de 7.5, réparties sur toute la
  période — le quantile gaussien 2.33σ est structurellement trop court (queues épaisses des
  résidus). **La dynamique est réparée, pas le niveau.**
- **Historique 250 j** : passe sur cette fenêtre (10 exceptions) — contrepoint honnête. Sa VaR,
  lue dans une queue empirique, était plus haute que la gaussienne avant le choc. Mais avec 10
  exceptions le test d'indépendance a peu de puissance, et le README montre qu'elle échoue à
  Kupiec sur l'échantillon complet 2014–2026.


In [3]:
vol_ewma = ewma_volatility(pnl, annualize=True) * 100
params_full = fit_garch(pnl.loc[:"2021-12-31"])
vol_garch = np.sqrt(garch_variance(pnl.loc[:"2021-12-31"], params_full)) * np.sqrt(252) * 100

fig, ax = plt.subplots(figsize=(11, 4.5))
abs_r = (pnl.loc[STUDY].abs() * 100)
ax.bar(abs_r.index, abs_r.to_numpy(), width=1.2, color="lightgray", label="|r_t| quotidien (%)")
ax.plot(vol_ewma.loc[STUDY].index, vol_ewma.loc[STUDY] / np.sqrt(252),
        color="tab:orange", lw=1.4, label="σ_t EWMA (journalière, %)")
ax.plot(vol_garch.loc[STUDY].index, vol_garch.loc[STUDY] / np.sqrt(252),
        color="tab:red", lw=1.2, ls="--", label="σ_t GARCH(1,1) (journalière, %)")
ax.set_title("Le clustering de volatilité, et deux filtres qui le suivent (2019–2021)")
ax.set_ylabel("%")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(ASSETS / "etude_2020_vol.png", dpi=110)
plt.show()
print("Paramètres GARCH (fit 2014→2021) :",
      f"omega={params_full.omega:.2e} alpha={params_full.alpha:.3f} beta={params_full.beta:.3f}",
      f"persistance={params_full.persistence:.3f}",
      f"vol LT annualisée={np.sqrt(params_full.long_run_variance * 252) * 100:.1f}%")


Paramètres GARCH (fit 2014→2021) : omega=5.94e-06 alpha=0.115 beta=0.845 persistance=0.961 vol LT annualisée=19.5%


Le graphe montre exactement ce que Christoffersen mesure : les |r_t| extrêmes arrivent par
paquets (mars 2020, pas isolément), et σ_t (EWMA comme GARCH) monte **avec** le paquet — là où
un σ de fenêtre 250 j resterait plat pendant des semaines.


In [4]:
zoom = slice("2019-10-01", "2020-12-31")
losses = -pnl.loc[zoom] * 100

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(losses.index, losses, color="lightsteelblue", lw=0.9, label="perte réalisée (%)")
colors = {"paramétrique 250j": "tab:blue", "EWMA (lambda=0.94)": "tab:orange",
          "GARCH(1,1) refit 20j": "tab:red"}
for name, color in colors.items():
    v = var_models[name].loc[zoom] * 100
    ax.plot(v.index, v, color=color, lw=1.6, label=f"VaR 99 % {name}")
    exc = exceptions_by_model[name]
    exc_dates = exc[exc == 1].loc[zoom].index
    ax.scatter(exc_dates, losses.loc[exc_dates], color=color, s=45, zorder=3, marker="v")

ax.set_title("Mars 2020 : la VaR conditionnelle suit le choc, l'inconditionnelle le subit")
ax.set_ylabel("% du portefeuille")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(ASSETS / "etude_2020_backtest.png", dpi=110)
plt.show()


Les triangles marquent les exceptions de chaque modèle. La ligne bleue (paramétrique 250 j)
traverse la crise presque à plat autour de 4 % — percée 9 fois en six semaines — puis reste
gonflée toute la fin 2020 alors que le calme est revenu (effet « fantôme » de la fenêtre : le
choc pèse tant qu'il n'est pas sorti des 250 jours). Les lignes EWMA/GARCH encaissent les 4
premiers jours du choc puis montent jusqu'à 12–15 % : plus aucune exception pendant le reste de
la crise, et une décrue rapide dès avril.


In [5]:
# Stabilité des paramètres GARCH au fil des réestimations (refit tous les 20 j, fenêtre 1000 j)
study_positions = [i for i in range(1000, len(pnl))
                   if pnl.index[i] >= pd.Timestamp("2019-01-01")
                   and pnl.index[i] <= pd.Timestamp("2021-12-31")]
refit_positions = [i for i in study_positions if (i - 1000) % 20 == 0]
trajectory = []
for i in refit_positions:
    p = fit_garch(pnl.iloc[i - 1000 : i])
    trajectory.append({"date": pnl.index[i], "alpha": p.alpha, "beta": p.beta,
                       "persistance": p.persistence})
trajectory = pd.DataFrame(trajectory).set_index("date")

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(trajectory.index, trajectory["alpha"], label="alpha (réaction)", color="tab:red")
ax.plot(trajectory.index, trajectory["beta"], label="beta (persistance)", color="tab:blue")
ax.plot(trajectory.index, trajectory["persistance"], label="alpha+beta", color="black", ls=":")
ax.axhline(1.0, color="gray", lw=0.8)
ax.set_title("Trajectoire des paramètres GARCH refités (2019–2021)")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(ASSETS / "etude_2020_garch_params.png", dpi=110)
plt.show()
trajectory.describe().loc[["min", "mean", "max"]].round(3)


,alpha,beta,persistance
min,0.086,0.778,0.926
mean,0.144,0.817,0.961
max,0.179,0.894,0.980


## Conclusion

1. **Le défaut mesuré en brique 0 est un défaut de dynamique** : Christoffersen rejette la VaR
   inconditionnelle parce que σ constant sur 250 jours ignore le clustering de volatilité.
2. **σ_t conditionnel (EWMA/GARCH) répare la dynamique** : indépendance p ≈ 0.61, exceptions
   étalées, VaR ×3 en quelques jours de crise.
3. **Il ne répare pas le niveau à 99 %** : 21 exceptions pour 7.5 attendues, Kupiec rejette —
   les résidus standardisés r_t/σ_t restent leptokurtiques, le quantile gaussien 2.33 est trop
   court. La suite logique est la **brique 2 : innovations Student-t** (et l'Expected Shortfall).
4. À noter : la persistance GARCH estimée frôle 1 (α+β ≈ 0.97–0.99) — les chocs de variance
   sont très persistants, ce qui justifie aussi l'EWMA (persistance 1 imposée) comme
   approximation de marché raisonnable.
